In [ ]:
from datetime import date
from pathlib import Path
import json
import math
import re
import time

import ee
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

GEE_PROJECT = "viirs-peru"
MB_ASSET = "projects/mapbiomas-public/assets/peru/collection4/mapbiomas_peru_collection4_coverage_v1"
ASSETS_EXTRA = [MB_ASSET]              # assets que el listado de GEE puede no devolver, pero que se leen por su ruta
ANIO_INI, ANIO_FIN = 2013, 2025        # periodo de interés (2025 = último año de la cobertura C4)
REFRESCAR_CACHE = False                # True = volver a consultar GEE aunque exista la caché local
MAX_BANDAS_MEDIR = 14                  # bandas por asset a medir (desde 2013, repartidas); 14 cubre los 13 años
BANDAS_POR_LLAMADA = 7                 # bandas por consulta a GEE (evita 'Too many concurrent aggregations')
MAX_IMAGENES_COLECCION = 24            # imágenes que se miden por cada colección de imágenes (mensuales)


def buscar_raiz(inicio=None):
    """Sube por las carpetas hasta encontrar la raíz del proyecto (la que tiene el límite provincial)."""
    inicio = Path(inicio or Path.cwd()).resolve()
    for carpeta in [inicio, *inicio.parents]:
        if (carpeta / "data" / "raw" / "Provincia" / "PROVINCIA.gpkg").is_file():
            return carpeta
    raise FileNotFoundError("No encuentro data/raw/Provincia/PROVINCIA.gpkg desde esta ubicación.")


def iniciar_gee(proyecto=GEE_PROJECT):
    try:
        ee.Initialize(project=proyecto)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=proyecto)


def con_reintentos(fn, intentos=5, espera=6):
    """Ejecuta fn() y reintenta con espera creciente si GEE responde con límites de concurrencia o cuota."""
    for i in range(intentos):
        try:
            return fn()
        except ee.EEException as e:
            texto = str(e)
            if i == intentos - 1 or not any(k in texto for k in ("concurrent", "Too many", "429", "uota", "timed out", "Computation")):
                raise
            time.sleep(espera * (i + 1))


iniciar_gee()
RAIZ = buscar_raiz()
PROC = RAIZ / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)

gdf = gpd.read_file(RAIZ / "data" / "raw" / "Provincia" / "PROVINCIA.gpkg")
prov = gdf[(gdf["ccdd"] == "08") & (gdf["ccpp"] == "01")]
assert len(prov) == 1, "El filtro de provincia no devolvió exactamente 1 registro"
aoi = ee.Geometry(prov.geometry.iloc[0].simplify(0.0001, preserve_topology=True).__geo_interface__)
AREA_PROV_KM2 = prov.to_crs(epsg=32719).area.iloc[0] / 1e6

print("Provincia:", prov["nombprov"].iloc[0], f"| {AREA_PROV_KM2:,.0f} km² | Raíz:", RAIZ)
CONFIG_LISTA = True   # marca que la configuración corrió; las demás secciones la comprueban

In [ ]:
if "CONFIG_LISTA" not in globals():
    raise RuntimeError("Falta la configuración: ejecuta primero la PRIMERA celda de código (o usa 'Run All'). "
                       "Si acabas de reiniciar el kernel, vuelve a ejecutarla.")

RUTA_CACHE = PROC / "_cache_mapbiomas_inventario_v2.json"


def listar_recursivo(carpeta, nivel=0, max_nivel=4):
    """Lista assets de GEE de forma recursiva (solo se entra en carpetas)."""
    salida, token = [], None
    while True:
        params = {"parent": carpeta}
        if token:
            params["pageToken"] = token
        try:
            resp = con_reintentos(lambda: ee.data.listAssets(params))
        except Exception as e:
            print(f"  no pude listar {carpeta}: {repr(e)[:140]}")
            return salida
        for a in resp.get("assets", []):
            salida.append({"id": a["id"], "tipo": a["type"], "nivel": nivel})
            if a["type"] == "FOLDER" and nivel < max_nivel:
                salida += listar_recursivo(a["id"], nivel + 1, max_nivel)
        token = resp.get("nextPageToken")
        if not token:
            return salida


CARPETA_PERU = MB_ASSET.rsplit("/", 2)[0]       # projects/mapbiomas-public/assets/peru
if RUTA_CACHE.is_file() and not REFRESCAR_CACHE:
    cache = json.loads(RUTA_CACHE.read_text(encoding="utf-8"))
    print("Caché cargada:", RUTA_CACHE.name, "| fecha:", cache.get("fecha"))
else:
    cache = {"fecha": pd.Timestamp.now().isoformat(timespec="seconds"), "assets": [], "info": {}, "colecciones": {}, "tablas": {}, "senal": {}}
    cache["assets"] = listar_recursivo(CARPETA_PERU)
    print(f"{CARPETA_PERU}: {len(cache['assets'])} assets listados desde GEE")

# Assets que el listado puede no devolver (p. ej. la cobertura C4): se añaden por su ruta
ids = {a["id"] for a in cache["assets"]}
for extra in ASSETS_EXTRA:
    if extra not in ids:
        try:
            tipo = con_reintentos(lambda: ee.data.getAsset(extra)["type"])
            cache["assets"].append({"id": extra, "tipo": tipo, "nivel": -1, "extra": True})
            print("Añadido por ruta:", extra.split("/")[-1], f"({tipo})")
        except Exception as e:
            print("⚠ No pude leer el asset extra", extra, repr(e)[:140])


def familia_de(n, tipo):
    n = n.lower()
    if tipo == "TABLE":
        return "fuego" if "burned" in n else "otros"
    if "burned" in n or "fire" in n:
        return "fuego"
    if "transition" in n:
        return "transiciones"
    if "water" in n:
        return "agua"
    if "quality" in n:
        return "calidad/frecuencia"
    if "secondary_vegetation_age" in n:
        return "vegetación secundaria"
    if "deforestation" in n:
        return "deforestación"
    if "integration" in n or "coverage" in n:
        return "cobertura"
    return "otros"


def producto_de(n):
    """Nombre del producto sin colección ni versión (para comparar versiones del mismo producto)."""
    k = re.sub(r"^mapbiomas_", "", n)
    k = re.sub(r"^peru_", "", k)
    k = re.sub(r"collection\d+_?", "", k)
    k = re.sub(r"_v\d+$", "", k).strip("_")
    return "coverage" if k == "integration" else k


def numero(patron, texto):
    m = re.search(patron, texto)
    return int(m.group(1)) if m else np.nan


filas = []
for a in cache["assets"]:
    if a["tipo"] == "FOLDER":
        continue
    n = a["id"].split("/")[-1]
    filas.append({"asset": a["id"], "nombre": n, "tipo": a["tipo"], "familia": familia_de(n, a["tipo"]),
                  "producto": producto_de(n) if a["tipo"] != "TABLE" else "annual_burned_table",
                  "coleccion": numero(r"collection(\d+)", a["id"]), "version": numero(r"_v(\d+)$", n)})
inv = pd.DataFrame(filas)
inv["es_tabla"] = inv["tipo"].eq("TABLE")
inv["es_coleccion"] = inv["tipo"].eq("IMAGE_COLLECTION")

# Última versión de cada producto: SOLO informativa
img = inv[~inv["es_tabla"]].copy()
img["_orden"] = list(zip(img["coleccion"].fillna(0), img["version"].fillna(0)))
mejor = img.groupby("producto")["_orden"].transform("max")
inv["ultima_version"] = False
inv.loc[img.index[img["_orden"] == mejor], "ultima_version"] = True

print(f"Assets: {len(inv)} | imágenes: {int(inv['tipo'].eq('IMAGE').sum())} | colecciones de imágenes: {int(inv['es_coleccion'].sum())} | tablas: {int(inv['es_tabla'].sum())}")
print("\nPor familia:")
print(inv.groupby(["familia", "tipo"]).size().to_string())
print("\nPor colección:")
print(inv.groupby(["coleccion"], dropna=False).size().to_string())
inv[~inv["es_tabla"]][["familia", "producto", "tipo", "coleccion", "version", "ultima_version", "nombre"]].sort_values(["familia", "producto", "coleccion", "version"])

In [ ]:
if "CONFIG_LISTA" not in globals():
    raise RuntimeError("Falta la configuración: ejecuta primero la PRIMERA celda de código (o usa 'Run All'). "
                       "Si acabas de reiniciar el kernel, vuelve a ejecutarla.")

RE_MES = re.compile(r"((?:19|20)\d{2})_(0[1-9]|1[0-2])$")
RE_DOS = re.compile(r"((?:19|20)\d{2})_((?:19|20)\d{2})$")
RE_UNO = re.compile(r"((?:19|20)\d{2})$")


def anio_banda(b):
    """Año asociado a una banda: el final en pares AAAA_AAAA, el año en AAAA_MM o AAAA; None si no hay año."""
    for patron, g in ((RE_MES, 1), (RE_DOS, 2), (RE_UNO, 1)):
        m = patron.search(b)
        if m:
            return int(m.group(g))
    return None


def analizar_bandas(bandas):
    """Resolución temporal y años asociados según los nombres de banda."""
    meses, dos, uno = [], [], []
    for b in bandas:
        m, d, u = RE_MES.search(b), RE_DOS.search(b), RE_UNO.search(b)
        if m:
            meses.append((int(m.group(1)), int(m.group(2))))
        elif d:
            dos.append((int(d.group(1)), int(d.group(2))))
        elif u:
            uno.append(int(u.group(1)))
    if meses:
        res, anios = "mensual", sorted({a for a, _ in meses})
    elif dos:
        consecutivos = [p for p in dos if p[1] - p[0] == 1]
        largos = [p for p in dos if p[1] - p[0] > 1]
        if len(dos) == 1:
            res, anios = "sin eje temporal", []
        elif consecutivos and largos:
            res, anios = "mixto (anual + acumulado)", sorted({b for _, b in dos})
        elif consecutivos:
            res, anios = "anual", sorted({b for _, b in dos})
        else:
            res, anios = "acumulado", sorted({b for _, b in dos})
    elif uno:
        res, anios = "anual", sorted(set(uno))
    else:
        res, anios = "sin eje temporal", []
    return {"resolucion": res, "anios": anios, "n_bandas": len(bandas)}


def analizar_coleccion(c):
    """Resolución temporal de una colección de imágenes según las fechas de sus imágenes."""
    fechas = pd.to_datetime(c.get("fechas", []))
    if len(fechas) == 0:
        return {**analizar_bandas(c["bandas"]), "resolucion": "desconocida (colección sin fechas)"}
    df = pd.DataFrame({"y": fechas.year, "m": fechas.month})
    por_anio = df.groupby("y")["m"].nunique()
    res = "mensual (colección)" if por_anio.median() > 1 else "anual (colección)"
    return {"resolucion": res, "anios": sorted({int(y) for y in df["y"]}), "n_bandas": len(c["bandas"])}


# --- Información de cada imagen / colección (GEE, con caché) ---
def info_imagen(asset):
    im = ee.Image(asset)
    d = im.getInfo()
    bandas = [b["id"] for b in d["bands"]]
    prec = {b["id"]: b["data_type"].get("precision", "") for b in d["bands"]}
    esc = im.select(0).projection().nominalScale().getInfo()
    return {"bandas": bandas, "precision": prec, "esc": float(esc)}


def info_coleccion(asset):
    col = ee.ImageCollection(asset)
    n = col.size().getInfo()
    if n == 0:
        return {"n": 0, "bandas": [], "fechas": [], "precision": {}, "esc": 30.0}
    primera = ee.Image(col.first())
    d = primera.getInfo()
    bandas = [b["id"] for b in d["bands"]]
    prec = {b["id"]: b["data_type"].get("precision", "") for b in d["bands"]}
    esc = float(primera.select(0).projection().nominalScale().getInfo())
    marcas = col.aggregate_array("system:time_start").getInfo()
    fechas = [pd.Timestamp(m, unit="ms").strftime("%Y-%m-%d") for m in marcas if m is not None]
    return {"n": n, "bandas": bandas, "fechas": fechas, "precision": prec, "esc": esc}


for r in inv.itertuples():
    if r.es_tabla:
        continue
    destino = cache["colecciones"] if r.es_coleccion else cache["info"]
    if r.asset in destino:
        continue
    try:
        destino[r.asset] = con_reintentos(lambda: info_coleccion(r.asset) if r.es_coleccion else info_imagen(r.asset))
    except Exception as e:
        destino[r.asset] = {"error": repr(e)[:200]}

# --- Tablas: elementos dentro de la provincia y campos (solo años desde ANIO_INI) ---
for asset in inv.loc[inv["es_tabla"], "asset"]:
    m = RE_UNO.search(asset.split("/")[-1])
    if asset in cache["tablas"] or not m or int(m.group(1)) < ANIO_INI:
        continue
    try:
        fc = ee.FeatureCollection(asset)
        n_prov = con_reintentos(lambda: fc.filterBounds(aoi).size().getInfo())
        campos = con_reintentos(lambda: ee.Feature(fc.first()).propertyNames().getInfo())
        cache["tablas"][asset] = {"en_provincia": n_prov, "campos": campos}
    except Exception as e:
        cache["tablas"][asset] = {"error": repr(e)[:160]}

RUTA_CACHE.write_text(json.dumps(cache, ensure_ascii=False, indent=1), encoding="utf-8")

filas = []
for r in inv.itertuples():
    base = {"asset": r.asset, "error": "", "elementos_en_provincia": np.nan, "esc": np.nan, "continuo": False, "n_imagenes": np.nan}
    if r.es_tabla:
        t = cache["tablas"].get(r.asset)
        anio = numero(r"(\d{4})$", r.nombre)
        filas.append({**base, "resolucion": "anual (tabla, un archivo por año)" if t else "(no consultada)",
                      "anios": [int(anio)] if (t and anio >= ANIO_INI) else [], "n_bandas": np.nan,
                      "error": t.get("error", "") if t else "", "elementos_en_provincia": t.get("en_provincia") if t else np.nan})
        continue
    i = (cache["colecciones"] if r.es_coleccion else cache["info"]).get(r.asset, {"error": "sin información"})
    if "error" in i:
        filas.append({**base, "resolucion": "error", "anios": [], "n_bandas": np.nan, "error": i["error"]})
        continue
    a = analizar_coleccion(i) if r.es_coleccion else analizar_bandas(i["bandas"])
    prim = i["bandas"][0] if i["bandas"] else ""
    filas.append({**base, **a, "esc": i["esc"], "continuo": i["precision"].get(prim, "") == "float",
                  "n_imagenes": i.get("n", np.nan)})
res_df = pd.DataFrame(filas)
cat = inv.merge(res_df, on="asset")
cat["años_serie"] = cat["anios"].apply(lambda L: [a for a in L if ANIO_INI <= a <= ANIO_FIN])
cat["años_2013_2025"] = cat["años_serie"].apply(lambda L: f"{min(L)}–{max(L)} ({len(L)})" if L else "ninguno")

print("Resolución temporal detectada:")
print(cat.groupby(["resolucion"]).size().to_string())
errores = cat[cat["resolucion"] == "error"]
if len(errores):
    print("\nAssets que no se pudieron leer:")
    print(errores[["nombre", "tipo", "error"]].to_string())
cat[["familia", "producto", "tipo", "coleccion", "version", "resolucion", "n_bandas", "n_imagenes", "años_2013_2025", "continuo"]].sort_values(["familia", "producto", "coleccion", "version"])

In [ ]:
def elegir_bandas(bandas):
    """Bandas con año entre ANIO_INI y ANIO_FIN, repartidas hasta MAX_BANDAS_MEDIR."""
    cand = sorted((anio_banda(b), b) for b in bandas if anio_banda(b) and ANIO_INI <= anio_banda(b) <= ANIO_FIN)
    if len(cand) > MAX_BANDAS_MEDIR:
        idx = sorted(set(np.linspace(0, len(cand) - 1, MAX_BANDAS_MEDIR).round().astype(int)))
        cand = [cand[i] for i in idx]
    return [b for _, b in cand]


def filas_de_histograma(asset, banda, anio, h, esc):
    if not h:
        return [{"asset": asset, "banda": banda, "año": anio, "valor": np.nan, "pixeles": 0, "km2": 0.0}]
    return [{"asset": asset, "banda": banda, "año": anio, "valor": float(v), "pixeles": cnt, "km2": cnt * esc * esc / 1e6}
            for v, cnt in h.items()]


def medir_imagen(asset, bandas, esc, continuo):
    """Cuenta píxeles por valor (o media y máximo si es continuo) dentro de la provincia, por lotes de bandas."""
    filas = []
    for k in range(0, len(bandas), BANDAS_POR_LLAMADA):
        lote = bandas[k:k + BANDAS_POR_LLAMADA]
        im = ee.Image(asset).select(lote)
        red = (ee.Reducer.mean().combine(ee.Reducer.max(), sharedInputs=True) if continuo else ee.Reducer.frequencyHistogram())
        res = con_reintentos(lambda: im.reduceRegion(red, aoi, esc, maxPixels=1e13).getInfo())
        for b in lote:
            if continuo:
                filas.append({"asset": asset, "banda": b, "año": anio_banda(b), "valor": np.nan, "pixeles": np.nan, "km2": np.nan,
                              "media": res.get(f"{b}_mean"), "maximo": res.get(f"{b}_max")})
            else:
                filas += filas_de_histograma(asset, b, anio_banda(b), res.get(b), esc)
    return filas


def medir_coleccion(asset, info):
    """Mide hasta MAX_IMAGENES_COLECCION imágenes (2013–2025, repartidas) de una colección de imágenes."""
    fechas = [(i, f) for i, f in enumerate(info["fechas"]) if ANIO_INI <= int(f[:4]) <= ANIO_FIN]
    if len(fechas) > MAX_IMAGENES_COLECCION:
        idx = sorted(set(np.linspace(0, len(fechas) - 1, MAX_IMAGENES_COLECCION).round().astype(int)))
        fechas = [fechas[i] for i in idx]
    col = ee.ImageCollection(asset).sort("system:time_start")
    lista = col.toList(info["n"])
    prim = info["bandas"][0]
    continuo = info["precision"].get(prim, "") == "float"
    filas = []
    for i, f in fechas:
        im = ee.Image(lista.get(i)).select(prim)
        red = (ee.Reducer.mean().combine(ee.Reducer.max(), sharedInputs=True) if continuo else ee.Reducer.frequencyHistogram())
        res = con_reintentos(lambda: im.reduceRegion(red, aoi, info["esc"], maxPixels=1e13).getInfo())
        banda = f"{prim}@{f}"
        if continuo:
            filas.append({"asset": asset, "banda": banda, "año": int(f[:4]), "valor": np.nan, "pixeles": np.nan, "km2": np.nan,
                          "media": res.get(f"{prim}_mean"), "maximo": res.get(f"{prim}_max")})
        else:
            filas += filas_de_histograma(asset, banda, int(f[:4]), res.get(prim), info["esc"])
    return filas

In [ ]:
medibles = cat[(~cat["es_tabla"]) & (cat["resolucion"] != "error")]
pend = [a for a in medibles["asset"] if a not in cache["senal"]]
print(f"Assets a medir en la provincia: {len(pend)} (el resto ya está en la caché)")
for k, asset in enumerate(pend, start=1):
    fila = medibles[medibles["asset"] == asset].iloc[0]
    try:
        if fila["es_coleccion"]:
            cache["senal"][asset] = medir_coleccion(asset, cache["colecciones"][asset])
        else:
            bandas = elegir_bandas(cache["info"][asset]["bandas"])
            cache["senal"][asset] = medir_imagen(asset, bandas, fila["esc"], fila["continuo"]) if bandas else []
    except Exception as e:
        cache["senal"][asset] = [{"asset": asset, "banda": "(error)", "año": np.nan, "valor": np.nan, "pixeles": np.nan, "km2": np.nan,
                                  "error": repr(e)[:160]}]
    print(f"  [{k}/{len(pend)}] {fila['nombre']}")
    if k % 5 == 0:
        RUTA_CACHE.write_text(json.dumps(cache, ensure_ascii=False, indent=1), encoding="utf-8")
RUTA_CACHE.write_text(json.dumps(cache, ensure_ascii=False, indent=1), encoding="utf-8")

senal_bandas = pd.DataFrame([f for L in cache["senal"].values() for f in L])
if "media" not in senal_bandas:
    senal_bandas["media"] = np.nan
    senal_bandas["maximo"] = np.nan
print("Filas de medición:", len(senal_bandas), "| assets con error de medición:",
      int(senal_bandas["banda"].eq("(error)").sum()))
senal_bandas.head(10)

In [ ]:
if "CONFIG_LISTA" not in globals():
    raise RuntimeError("Falta la configuración: ejecuta primero la PRIMERA celda de código (o usa 'Run All'). "
                       "Si acabas de reiniciar el kernel, vuelve a ejecutarla.")

mens_fuego = cat[(cat["familia"] == "fuego") & cat["nombre"].str.contains("monthly_burned") & ~cat["nombre"].str.contains("coverage")
                 & (cat["tipo"] == "IMAGE") & (cat["resolucion"] != "error")]
fuego_confirmado = set()
largos = []
for r in mens_fuego.itertuples():
    print("=" * 90)
    print(r.nombre, f"(C{int(r.coleccion)}, años {r.años_2013_2025})")
    sb = senal_bandas[(senal_bandas["asset"] == r.asset) & senal_bandas["valor"].notna() & (senal_bandas["año"] >= ANIO_INI)]
    sb = sb[sb["valor"] != 0].copy()
    if sb.empty:
        print("  sin píxeles con valor distinto de 0 en la provincia: no se puede contrastar.")
        continue
    sb["mes"] = sb["valor"].astype(int)
    print("  Valores de píxel observados:", sorted(int(v) for v in sb["mes"].unique()))
    c1 = set(sb["mes"]) <= set(range(1, 13))
    tabla = sb[sb["mes"].between(1, 12)].pivot_table(index="año", columns="mes", values="km2", aggfunc="sum").reindex(columns=range(1, 13)).fillna(0.0)

    asset_anual = r.asset.replace("monthly_burned", "annual_burned")
    anual = senal_bandas[(senal_bandas["asset"] == asset_anual) & (senal_bandas["valor"] == 1.0)].groupby("año")["km2"].sum()
    comp = pd.DataFrame({"suma 12 valores (km²)": tabla.sum(axis=1).round(3), "área quemada anual (km²)": anual.reindex(tabla.index).round(3)})
    comp["diferencia %"] = ((comp.iloc[:, 0] - comp.iloc[:, 1]).abs() / comp.iloc[:, 1].replace(0, np.nan) * 100).round(1)
    dif = comp["diferencia %"].dropna()
    c2 = bool(len(dif) > 0 and (dif <= 5).mean() >= 0.8)
    print(comp.to_string())
    estacion = tabla.loc[:, 7:11].sum().sum() / max(tabla.sum().sum(), 1e-9)
    c3 = estacion >= 0.7
    ok = bool(c1 and c2 and c3)
    print(f"  ¿Valores entre 1 y 12? {c1} | ¿suma ≈ área anual (≤5 % en ≥80 % de años)? {c2} (contraparte: {asset_anual.split('/')[-1]}) | "
          f"% del área en jul–nov: {estacion * 100:.0f} % → estacional {c3}")
    print("  →", "CONSISTENTE con valor = mes (por confirmar en la documentación)" if ok else "NO consistente: no usar como mes")
    if ok:
        fuego_confirmado.add(r.asset)
    largo = tabla.stack().rename("area_km2").reset_index()
    largo.insert(0, "coleccion", int(r.coleccion))
    largo.insert(0, "producto", r.nombre)
    largo["estado"] = ("consistente con que el valor sea el mes; falta confirmar en la documentación de MapBiomas Fuego" if ok
                       else "NO consistente: no usar como mes")
    largos.append(largo)

    fig, ejes = plt.subplots(1, 2, figsize=(13, 4.2), gridspec_kw={"width_ratios": [1.5, 1]})
    im = ejes[0].imshow(tabla.values, aspect="auto", cmap="YlOrRd")
    ejes[0].set_xticks(range(12), range(1, 13))
    ejes[0].set_yticks(range(len(tabla)), tabla.index)
    ejes[0].set_xlabel("mes (valor del píxel)")
    ejes[0].set_title(f"Área quemada por año y mes (km²) · C{int(r.coleccion)}")
    plt.colorbar(im, ax=ejes[0])
    ejes[1].bar(range(1, 13), tabla.sum(), color="#c0392b")
    ejes[1].set_xticks(range(1, 13))
    ejes[1].set_xlabel("mes")
    ejes[1].set_ylabel("km² acumulados")
    ejes[1].set_title("Estacionalidad")
    plt.tight_layout()
    plt.show()

fuego_mensual = pd.concat(largos, ignore_index=True) if largos else pd.DataFrame()
if not len(mens_fuego):
    print("No hay assets de fuego mensual legibles.")

In [ ]:
otros_mens = cat[cat["resolucion"].astype(str).str.startswith("mensual") & ~cat["asset"].isin(mens_fuego["asset"])]
if otros_mens.empty:
    print("No hay otros productos mensuales legibles.")
for r in otros_mens.itertuples():
    print("=" * 90)
    print(r.nombre, f"| tipo {r.tipo} | imágenes: {r.n_imagenes} | años {r.años_2013_2025}")
    info = cache["colecciones"].get(r.asset) or cache["info"].get(r.asset) or {}
    if info.get("fechas"):
        print("  fechas:", info["fechas"][0], "→", info["fechas"][-1], f"({len(info['fechas'])})")
    print("  bandas:", info.get("bandas", [])[:5])
    sb = senal_bandas[(senal_bandas["asset"] == r.asset) & senal_bandas["valor"].notna() & (senal_bandas["valor"] != 0)]
    print("  valores observados (muestra):", sorted({int(v) for v in sb["valor"]})[:15])
    por = sb.groupby("banda")["km2"].sum()
    print("  área con valor ≠ 0 por imagen medida (km²):")
    print(por.round(3).to_string())

# Productos con error de lectura que por su nombre podrían ser mensuales
sospechosos = cat[(cat["resolucion"] == "error") & cat["nombre"].str.contains("monthly")]
if len(sospechosos):
    print("\nPor su nombre serían mensuales, pero NO se pudieron leer:")
    print(sospechosos[["nombre", "tipo", "error"]].to_string())

In [ ]:
if "CONFIG_LISTA" not in globals():
    raise RuntimeError("Falta la configuración: ejecuta primero la PRIMERA celda de código (o usa 'Run All'). "
                       "Si acabas de reiniciar el kernel, vuelve a ejecutarla.")

anios = list(range(ANIO_INI, ANIO_FIN + 1))
ordenado = cat.sort_values(["familia", "producto", "coleccion", "version"]).reset_index(drop=True)
ordenado["etiqueta"] = (ordenado["familia"] + " · " + ordenado["producto"] + " · C" + ordenado["coleccion"].fillna(0).astype(int).astype(str)
                        + (" v" + ordenado["version"].fillna(0).astype(int).astype(str)).where(~ordenado["es_tabla"], ""))
matriz_disp = np.array([[a in set(L) for a in anios] for L in ordenado["años_serie"]], dtype=float)

filas_disp = [{"asset": r.asset, "familia": r.familia, "producto": r.producto, "coleccion": r.coleccion, "version": r.version,
               "año": a, "disponible": bool(a in set(r.años_serie))} for r in ordenado.itertuples() for a in anios]
disponibilidad = pd.DataFrame(filas_disp)

con_datos = ordenado[ordenado["años_serie"].apply(len) > 0].index
fig, ax = plt.subplots(figsize=(11, 0.28 * len(con_datos) + 1.5))
ax.imshow(matriz_disp[con_datos], aspect="auto", cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(anios)), anios, rotation=90)
ax.set_yticks(range(len(con_datos)), ordenado.loc[con_datos, "etiqueta"], fontsize=7)
ax.set_title("Años con datos por asset (azul = hay banda)")
plt.tight_layout()
plt.show()

sin_serie = ordenado[ordenado["años_serie"].apply(len) == 0]
print(f"Assets sin serie anual 2013–2025: {len(sin_serie)} (errores, productos de un solo periodo, tablas no consultadas o anteriores a 2013).")
print("Años cubiertos por familia (última versión):")
print(ordenado[ordenado["ultima_version"]].groupby("familia")["años_2013_2025"].agg(lambda s: " | ".join(sorted(set(s)))).to_string())

In [ ]:
if "CONFIG_LISTA" not in globals():
    raise RuntimeError("Falta la configuración: ejecuta primero la PRIMERA celda de código (o usa 'Run All'). "
                       "Si acabas de reiniciar el kernel, vuelve a ejecutarla.")

sb_ok = senal_bandas[senal_bandas["banda"] != "(error)"]
ev = sb_ok[sb_ok["valor"].notna() & (sb_ok["valor"] != 0)]
por_banda = ev.groupby(["asset", "banda"])["km2"].sum()
senal = por_banda.groupby("asset").agg(area_max_km2="max")
senal["años_con_señal"] = ev[ev["km2"] > 0].groupby("asset")["año"].nunique()
senal["valores_observados"] = ev.groupby("asset")["valor"].agg(lambda s: str(sorted({int(v) for v in s})[:12]))
senal["n_bandas_medidas"] = sb_ok.groupby("asset")["banda"].nunique()

cont = sb_ok[sb_ok["maximo"].notna()].groupby("asset").agg(media_max_continuo=("maximo", "max"))
senal = senal.join(cont, how="outer")
senal["n_bandas_medidas"] = senal["n_bandas_medidas"].fillna(sb_ok.groupby("asset")["banda"].nunique())

cat = cat.drop(columns=[c for c in cat.columns if c in senal.columns], errors="ignore").merge(senal, left_on="asset", right_index=True, how="left")
cat["pct_provincia"] = (cat["area_max_km2"] / AREA_PROV_KM2 * 100).round(3)
cat["pixeles_max"] = (cat["area_max_km2"] * 1e6 / (cat["esc"] ** 2)).round(0)
print("Assets con señal medida:", int(cat["area_max_km2"].notna().sum() + cat["media_max_continuo"].notna().sum()), "de", int((~cat["es_tabla"]).sum()), "imágenes/colecciones")
cat[~cat["es_tabla"]][["familia", "producto", "coleccion", "version", "area_max_km2", "pct_provincia", "pixeles_max", "años_con_señal", "valores_observados", "media_max_continuo"]] \
    .sort_values(["familia", "producto", "coleccion", "version"]).round(3)

In [ ]:
if "CONFIG_LISTA" not in globals():
    raise RuntimeError("Falta la configuración: ejecuta primero la PRIMERA celda de código (o usa 'Run All'). "
                       "Si acabas de reiniciar el kernel, vuelve a ejecutarla.")

PESOS = {"resolucion": 2, "cobertura_anios": 2, "señal": 3, "detectabilidad": 3, "comparabilidad": 2, "muestra": 2}
DETECTABILIDAD = {"cobertura": 5, "transiciones": 4, "deforestación": 4, "vegetación secundaria": 3, "agua": 4,
                  "fuego": 4, "calidad/frecuencia": 1, "otros": 1}        # juicio propio: sin verificar
COMP_COBERTURA = {4: 5, 3: 3, 2: 2, 1: 1}
UMBRAL_USAR, UMBRAL_RESERVAS = 65, 40
COL_MAX_FUEGO = cat.loc[cat["familia"] == "fuego", "coleccion"].max()

# La resolución efectiva del fuego mensual depende de la sección 3
cat["periodicidad"] = cat["resolucion"]
cat.loc[cat["asset"].isin(fuego_confirmado), "periodicidad"] = "mensual (mes codificado en el valor del píxel)"

NOTA_RES = {"mensual": 5, "anual": 3, "mixto": 2, "acumulado": 1, "sin": 0}


def nota_resolucion(p):
    for clave, v in NOTA_RES.items():
        if str(p).startswith(clave):
            return v
    return 0


def nota_senal(pct, media):
    if pd.isna(pct):
        return 2 if pd.notna(media) and media > 0 else 0
    return 5 if pct >= 1 else 4 if pct >= 0.1 else 3 if pct >= 0.01 else 2 if pct > 0 else 0


def nota_muestra(px):
    if pd.isna(px):
        return 0
    return 5 if px >= 1e5 else 4 if px >= 1e4 else 3 if px >= 1e3 else 2 if px >= 100 else 1 if px > 0 else 0


def nota_comparabilidad(r):
    if pd.isna(r.coleccion):
        return 1
    if r.familia == "fuego":
        return 5 if r.coleccion == COL_MAX_FUEGO else 2
    return COMP_COBERTURA.get(int(r.coleccion), 1)


puntaje = cat.copy()
puntaje["n_res"] = puntaje["periodicidad"].apply(nota_resolucion)
puntaje["n_anios"] = (puntaje["años_serie"].apply(len) / len(range(ANIO_INI, ANIO_FIN + 1)) * 5).clip(upper=5).round(2)
puntaje["n_senal"] = [nota_senal(a, b) for a, b in zip(puntaje["pct_provincia"], puntaje["media_max_continuo"])]
puntaje["n_det"] = puntaje["familia"].map(DETECTABILIDAD).fillna(1)
puntaje["n_comp"] = [nota_comparabilidad(r) for r in puntaje.itertuples()]
puntaje["n_muestra"] = puntaje["pixeles_max"].apply(nota_muestra)

w = PESOS
puntaje["puntaje"] = ((puntaje["n_res"] * w["resolucion"] + puntaje["n_anios"] * w["cobertura_anios"] + puntaje["n_senal"] * w["señal"]
                       + puntaje["n_det"] * w["detectabilidad"] + puntaje["n_comp"] * w["comparabilidad"]
                       + puntaje["n_muestra"] * w["muestra"]) / (5 * sum(w.values())) * 100).round(1)


def recomendar(r):
    if r.es_tabla:
        return "no puntuado (tabla vectorial)", "tabla de polígonos; no es una variable por píxel"
    if r.resolucion == "error" or str(r.resolucion).startswith("desconocida"):
        return "descartar", "error al leer el asset o resolución desconocida"
    if not r.años_serie and not str(r.resolucion).startswith("sin eje"):
        return "descartar", "sin años en 2013–2025"
    sin_senal = (pd.isna(r.area_max_km2) or r.area_max_km2 == 0) and not (pd.notna(r.media_max_continuo) and r.media_max_continuo > 0)
    if sin_senal:
        return "descartar", "sin señal en la provincia (o no se pudo medir)"
    if r.puntaje >= UMBRAL_USAR:
        return "usar", f"puntaje {r.puntaje}"
    if r.puntaje >= UMBRAL_RESERVAS:
        return "usar con reservas", f"puntaje {r.puntaje}"
    return "descartar", f"puntaje bajo ({r.puntaje})"


rec = [recomendar(r) for r in puntaje.itertuples()]
puntaje["recomendacion"] = [x[0] for x in rec]
puntaje["motivo"] = [x[1] for x in rec]
catalogo = puntaje.drop(columns=["anios"]).copy()

# Mejor versión de cada producto por puntaje (desempate: colección y versión más altas); nada se descarta por antigüedad
catalogo["mejor_de_su_producto"] = False
cand = catalogo[(~catalogo["es_tabla"]) & catalogo["puntaje"].notna() & ~catalogo["recomendacion"].eq("descartar")]
mejores = cand.sort_values(["puntaje", "coleccion", "version"], ascending=False).groupby("producto").head(1).index
catalogo.loc[mejores, "mejor_de_su_producto"] = True

vista = catalogo[~catalogo["es_tabla"]].sort_values("puntaje", ascending=False)
print(vista["recomendacion"].value_counts().to_string())
vista[["familia", "producto", "coleccion", "version", "periodicidad", "años_2013_2025", "pct_provincia", "puntaje", "recomendacion", "mejor_de_su_producto", "motivo"]]

In [ ]:
top = vista[vista["puntaje"].notna()].head(25).iloc[::-1]
fig, ax = plt.subplots(figsize=(10, 0.35 * len(top) + 1.5))
colores = top["recomendacion"].map({"usar": "#2e7d32", "usar con reservas": "#f9a825", "descartar": "#9e9e9e"}).fillna("#9e9e9e")
ax.barh(top["familia"] + " · " + top["producto"] + " · C" + top["coleccion"].fillna(0).astype(int).astype(str)
        + (" v" + top["version"].fillna(0).astype(int).astype(str)), top["puntaje"], color=colores)
ax.axvline(UMBRAL_USAR, color="#2e7d32", ls="--", lw=1)
ax.axvline(UMBRAL_RESERVAS, color="#f9a825", ls="--", lw=1)
ax.set_xlabel("puntaje (0–100)")
ax.set_title("Idoneidad como objetivo del modelo (todas las versiones; verde = usar, amarillo = reservas)")
plt.tight_layout()
plt.show()

In [ ]:
if "CONFIG_LISTA" not in globals():
    raise RuntimeError("Falta la configuración: ejecuta primero la PRIMERA celda de código (o usa 'Run All'). "
                       "Si acabas de reiniciar el kernel, vuelve a ejecutarla.")

AGREGAR = []     # p. ej. ["mapbiomas_peru_fire_collection1_monthly_burned_v1"]
QUITAR = []      # nombres de asset a excluir aunque estén recomendados
DERIVADOS_C4 = ["área por clase", "pérdida de bosque", "expansión agrícola", "expansión de bosque", "balance neto de bosque",
                "área estable", "área que cambió de clase", "bosque al inicio", "pérdida acumulada", "tasa de pérdida"]


def periodicidad_modelo(p):
    p = str(p)
    return "mensual" if p.startswith("mensual") else "acumulado" if p.startswith("acumulado") else "anual" if p.startswith(("anual", "mixto")) else "sin eje temporal"


# Todas las versiones de cada producto, lado a lado
versiones = (catalogo[~catalogo["es_tabla"] & catalogo["puntaje"].notna()]
             .assign(v=lambda d: "C" + d["coleccion"].fillna(0).astype(int).astype(str) + "v" + d["version"].fillna(0).astype(int).astype(str)
                     + ":" + d["puntaje"].astype(str) + np.where(d["mejor_de_su_producto"], "★", ""))
             .groupby(["familia", "producto"])["v"].agg(" | ".join))
print("Versiones por producto (★ = mejor por puntaje):")
print(versiones.to_string())

base = catalogo[(~catalogo["es_tabla"]) & catalogo["recomendacion"].isin(["usar", "usar con reservas"]) & catalogo["mejor_de_su_producto"]]
elegidos = pd.concat([base, catalogo[catalogo["nombre"].isin(AGREGAR)]]).drop_duplicates("asset")
elegidos = elegidos[~elegidos["nombre"].isin(QUITAR)].copy()
elegidos["tipo_fila"] = "asset de MapBiomas"
elegidos["periodicidad_modelo"] = elegidos["periodicidad"].apply(periodicidad_modelo)
INDICADORES_ELEGIDOS = elegidos["nombre"].tolist()

derivados = pd.DataFrame({"nombre": DERIVADOS_C4}).assign(
    tipo_fila="derivado de la cobertura C4 (se calcula en el notebook 06)", familia="cobertura", periodicidad_modelo="anual",
    años_2013_2025="2013–2025", unidades="km², % provincia, %/año", recomendacion="usar",
    motivo="se obtiene sumando áreas del mapa de cobertura C4")

elegidos_df = pd.concat([
    elegidos[["tipo_fila", "nombre", "familia", "coleccion", "version", "periodicidad_modelo", "periodicidad", "años_2013_2025", "puntaje", "recomendacion", "motivo"]],
    derivados,
], ignore_index=True)

print("\nINDICADORES_ELEGIDOS (assets):", len(INDICADORES_ELEGIDOS))
for per in ("mensual", "anual", "acumulado"):
    sub = elegidos_df[elegidos_df["periodicidad_modelo"] == per]
    print(f"\n{per.upper()} ({len(sub)})")
    for r in sub.itertuples():
        print("  -", r.nombre, "|", r.familia, "|", r.años_2013_2025)
elegidos_df

In [ ]:
if "CONFIG_LISTA" not in globals():
    raise RuntimeError("Falta la configuración: ejecuta primero la PRIMERA celda de código (o usa 'Run All'). "
                       "Si acabas de reiniciar el kernel, vuelve a ejecutarla.")

catalogo_out = catalogo.copy()
catalogo_out["años_serie"] = catalogo_out["años_serie"].apply(lambda L: ",".join(map(str, L)))
catalogo_out.to_csv(PROC / "indicadores_catalogo_v2.csv", index=False, encoding="utf-8-sig")
elegidos_df.to_csv(PROC / "indicadores_elegidos_v2.csv", index=False, encoding="utf-8-sig")
disponibilidad.to_csv(PROC / "indicadores_disponibilidad_anio_v2.csv", index=False, encoding="utf-8-sig")
senal_bandas.to_csv(PROC / "indicadores_senal_bandas_v2.csv", index=False, encoding="utf-8-sig")
guardados = ["indicadores_catalogo_v2.csv", "indicadores_elegidos_v2.csv", "indicadores_disponibilidad_anio_v2.csv", "indicadores_senal_bandas_v2.csv"]
if len(fuego_mensual):
    fuego_mensual.to_csv(PROC / "fuego_mensual_v2.csv", index=False, encoding="utf-8-sig")
    guardados.append("fuego_mensual_v2.csv")
else:
    print("fuego_mensual_v2.csv no se guardó (la sección 3 no pudo calcularlo).")
for g in guardados:
    print("guardado:", PROC / g, f"({(PROC / g).stat().st_size / 1e3:.0f} KB)")

In [ ]:
if "CONFIG_LISTA" not in globals():
    raise RuntimeError("Falta la configuración: ejecuta primero la PRIMERA celda de código (o usa 'Run All'). "
                       "Si acabas de reiniciar el kernel, vuelve a ejecutarla.")

c = catalogo[~catalogo["es_tabla"]]
print("RESUMEN (calculado con los datos de esta corrida)")
print(f"- Assets de MapBiomas Perú en GEE: {len(inv)} ({int(inv['tipo'].eq('IMAGE').sum())} imágenes, "
      f"{int(inv['es_coleccion'].sum())} colecciones de imágenes, {int(inv['es_tabla'].sum())} tablas).")
print("- Resolución temporal:", c["resolucion"].value_counts().to_dict())
print("- Recomendación (todas las versiones):", c["recomendacion"].value_counts().to_dict())
print("- Fuego mensual consistente con valor = mes:", sorted(a.split("/")[-1] for a in fuego_confirmado) or "ninguno")
mens = elegidos_df[elegidos_df["periodicidad_modelo"] == "mensual"]["nombre"].tolist()
anu = elegidos_df[elegidos_df["periodicidad_modelo"] == "anual"]
print("- Indicadores MENSUALES elegidos:", mens or "ninguno")
print("- Indicadores ANUALES elegidos:", len(anu))
print("- Estadísticas territoriales: no disponibles en GEE (solo plataforma web).")
leidos_mal = c[c["resolucion"] == "error"]["nombre"].tolist()
print("- Assets que no se pudieron leer:", leidos_mal or "ninguno")
print("\nPENDIENTES / SIN VERIFICAR")
print("- Confirmar en la documentación de MapBiomas Fuego que el valor de `burned_monthly_AAAA` es el mes.")
print("- Los puntajes de `detectabilidad` son un juicio propio (DETECTABILIDAD); conviene revisarlos.")
print("- Las colecciones 1 a 3 pueden tener otra leyenda o algoritmo que la C4: comparabilidad sin verificar.")
print("- El área medida es aproximada (conteo de píxeles × superficie nominal del píxel).")
print("- Los productos con muchas bandas se miden en una muestra repartida de ≤", MAX_BANDAS_MEDIR, "bandas; las colecciones, en ≤", MAX_IMAGENES_COLECCION, "imágenes.")
print("- Nombres de clase de varios productos: sin verificar.")
print("- Ventana de meses del mosaico y clasificador de la C4 (ATBD): por confirmar.")
print("- Citar a MapBiomas (CC BY 4.0) en cualquier producto derivado.")